### 当代人工智能       $\ \ $   Lab01

##### 学号：10245501420  姓名：庞毅飞

#### 一、实验内容：

将提供的文本十分类数据集进行划分、预处理、特征转换后，使用四种不同的模型在训练集上进行训练，在验证集上选择最优方案，最后在测试集上进行分类预测，横向对比不同模型的最终效果。

#### 二、具体过程：

- 数据集划分、预处理：\
先检查文本是否缺失、标签是否为整数 0～9，并保存原始行号。然后采用随机种子42对数据划分以便于实验复现。这里训练集与验证集数据划分比例为4：1；然后清洗文本，处理换行和空白的问题，保留主题、括号等相关内容。在该任务中不读取测试集，防止“机器作弊”。

- 数据向量化——TF-IDF方法：\
预处理后，参考draft_main.py中提供的方法,创建一个向量化器将训练集和验证集中处理过的文本。这里我们采用5000维的TF-IDF，最终每一个文本对应一个高维稀疏向量。注意，fit过程会让模型学到词表、文档频率和 IDF信息，因此不能让验证集文本参与fit。

- 建立模型：\
在本次实验中，我选择了支持向量机（SVM）和多层感知机（MLP）两种算法完成分类模型的建立。\
首先，根据指导文件提示，我利用sklearn库中的线性核SVC对训练集进行升维+超平面分割操作来初步训练模型。初始时，我使用C=10作为默认参数，发现训练集准确率高达100%，但验证集准确率只有约90%，显然存在过拟合情况，于是逐步调整C的值，最终选取C=1时的训练模型为选定模型（此时验证集准确率91.5%）。同时，我也尝试使用高斯核来处理文本数据，发现训练过程明显变慢且参数调控很困难，没有筛选出理想的模型，因此最终仍采用线性核。\
接着，我开始构建多层感知机完成分类任务。同样利用sklearn库中已有的模块构建模型，但将训练的梯度下降算法改为小批量梯度下降算法（MBGD），每轮打乱训练样本，按默认batch=128条/批计算损失和梯度，执行 `参数 − 学习率 × 梯度 → 参数` $(\theta-\eta\Delta → \theta)$这一更新公式，尾批也参与更新。这样既能加快模型的训练速度，也能减少一定随机性，让梯度下降的方向总体朝着最优情况前进。在超参数调整过程中，我在固定batch=128的条件下，对学习率、训练轮数、正则化参数以及隐藏层规模进行控制变量形式的调整（固定其他参数，只改变一个参数，观察最终收敛时train_loss和validation_loss的变化情况，决定参数的调整方向），同时也测试了不同。最终，我选择了hidden=200，learning_rate=1.0，regularization=0.0001，epochs=46/150时的模型，此时验证集准确率约为92.4%。该模型验证集正确率高于SVM算法下的模型的正确率，最终选择这一模型进入测试。在mlp训练过程中，loss随epoch变化的曲线如下（报告有限，其余图片放在仓库中）：

<img src="./outputs/figures/mlp_loss.png" alt="loss曲线图" width="60%">

- 最终测试\
最后读取冻结配置，将全部7368条带标签文本用于重新拟合TF-IDF和模型，按选定的46轮重训，再转换并预测测试文本。输出2457行、无表头、无索引的整数标签CSV文件，并检查行序、合法标签及重载模型预测一致性。
#### 三、问题思考：

- 问题一：为什么SVM和逻辑回归适合这个数据集？\
TF-IDF有5000维，但训练矩阵超过90%的元素为0。不同类别往往可以通过少量主题词及其加权组合区分，线性模型直接学习这些词的正负权重，已能建立有效边界。稀疏运算降低计算成本，正则化则抑制高维特征带来的过拟合。高维并不保证线性可分，但本实验接近MLP的成绩、远低于MLP的时间复杂度和运算成本说明SVM算法下线性边界已经捕获了大部分有效信号，更适合这个数据集。相较于SVC的一对一求解的仍有较大开销，逻辑回归甚至可能更加高效。
- 问题二：增加MLP层数、神经元或epochs能否提高性能？\
我认为可能，但不能保证。当前`(50,)` 、`(100,)` 、`(100,50)` 的验证准确率分别为91.04%、91.18%、90.64%，没有呈现“越大越好”；100神经元模型已有500000多个参数 ，而训练样本仅5894条，很可能导致过拟合。虽然`max_iter=300`是上限，但是最佳验证下的MLP实际只训练到83轮epoch。即使模型还能继续训练下去，仍会因训练损失改善不足而停止，单纯把上限增至1000可能没有变化。不过，我们仍有可能通过上述行为进一步降低loss值。
- 问题三：除了TF-IDF，还有哪些文本表示？

| 方法 | 特点与局限 |
| --- | --- |
| 词袋模型BoW | 直接看使用词频/是否出现，简单高效，但忽略语序和语义相似性 |
| 词/字符n-gram | 保留局部顺序，字符表示对拼写变化、技术词较友好，但维度可能增大 |
| Word2Vec、GloVe、FastText | 稠密静态词向量，可平均或加权构成文档向量；FastText利用子词，但静态向量难区分一词多义 |
| LSA、Doc2Vec | 分别通过矩阵降维或文档向量学习得到紧凑表示 |
| BERT、Sentence-BERT等 | 提供上下文或句级语义表示，可冻结提取特征或微调；成本较高，长文本需截断或分块 |